# Advanced Options

Beyond the basic `vorpal build book.epub`, there are a number of optional passes and quality controls. This notebook demonstrates the most useful ones:

- **Draft mode** (`--draft`) — fast CPU synthesis for previewing structure before GPU run
- **Loudness profiles** (`--profile`) — target different listening environments
- **ASR round-trip check** (`--asr-check`) — Whisper verifies synthesis quality
- **Pronunciation lexicon** (`--lexicon`) — fix mispronounced proper nouns
- **Expressive mode** (`--expressive`) — LLM tone tagging for subtle delivery variation
- **Export** (`vorpal export`) — produce a clean text edition alongside the audiobook
- **Fidelity check** (`vorpal fidelity`) — verify no body text was silently dropped

**Prerequisites:** vorpal installed, ffmpeg on PATH. ASR check requires `pip install -e ".[audio]"`.

In [ ]:
import subprocess
import json
import pathlib
import urllib.request

# Download a short EPUB if we don't have one yet
epub_path = pathlib.Path("book.epub")
urllib.request.urlretrieve(
    "https://www.gutenberg.org/ebooks/35.epub.noimages", epub_path
)
print(f"Ready: {epub_path} ({epub_path.stat().st_size / 1024:.0f} KB)")

## Draft mode — fast preview on CPU

`--draft` switches to Piper, a lightweight CPU-only TTS engine. Synthesis is
near real-time on CPU but lower quality than Kokoro. Use it to check chapter
structure, chapter titles, and narration pacing before committing to a GPU run.

Draft and production workdirs are identical in structure. Switching from `--draft`
to a full build on the next run re-synthesises only the draft chunks.

In [ ]:
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--draft", "--end-page", "30", "--output", "book_draft"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(f"Return code: {result.returncode}")

## Loudness profiles

Three profiles target different listening environments:

| Profile | Target LUFS | When to use |
|---------|-------------|-------------|
| `headphones` | −18 LUFS | Default. Quiet environments. |
| `car` | −16 LUFS | Background road noise. |
| `speaker` | −20 LUFS | More headroom for room acoustics. |

In [ ]:
# Build with car profile — louder to cut through road noise
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--draft", "--end-page", "30",
     "--profile", "car", "--output", "book_car"],
    capture_output=True, text=True
)
print(result.stdout[-1000:])
print(f"Return code: {result.returncode}")

## Pronunciation lexicon

The `--lexicon` pass extracts unusual proper nouns and technical terms, then
proposes IPA phoneme overrides. The proposals are written to `lexicon.json`
in the workdir for you to review and correct.

After editing `lexicon.json`, a rebuild applies the corrected pronunciations.

In [ ]:
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--stop-after", "segment",
     "--lexicon", "--output", "book_lex"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(f"Return code: {result.returncode}")

# Show the extracted lexicon proposals
lexicon_path = pathlib.Path("book_lex_workdir") / "lexicon.json"
if lexicon_path.exists():
    proposals = json.loads(lexicon_path.read_text(encoding="utf-8"))
    print(f"\n{len(proposals)} lexicon proposals:")
    for term, entry in list(proposals.items())[:10]:
        print(f"  {term}: {entry}")

## ASR round-trip quality check

After synthesis, `--asr-check` runs a Whisper model on a sample of the output
audio and compares the transcript to the source text. Any systematic
mispronunciations or synthesis artefacts appear as character-level mismatches.

Requires `pip install -e ".[audio]"`.

In [ ]:
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--draft", "--end-page", "30",
     "--asr-check", "--asr-fraction", "0.20", "--output", "book_asr"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(f"Return code: {result.returncode}")

## Fidelity check

`vorpal fidelity` verifies that no body text was silently dropped during
normalisation. It compares the normalised text sent to the TTS engine against
the source, character by character. A score of 1.000 means everything made it through.

Only meaningful for EPUB and TXT sources — scanned PDFs have inherent OCR approximations.

In [ ]:
result = subprocess.run(
    ["vorpal", "fidelity", "--source", "book.epub", "--workdir", "book_draft_workdir"],
    capture_output=True, text=True
)
print(result.stdout)

## Export — clean text edition

`vorpal export` produces a clean EPUB or plain-text file from the normalised
pipeline output — boilerplate removed, OCR corrected (if `--repair` was used),
footnotes separated. Useful for producing a reading edition alongside the audiobook.

In [ ]:
result = subprocess.run(
    ["vorpal", "export", "book.epub", "--format", "txt", "--output", "book_clean.txt"],
    capture_output=True, text=True
)
print(result.stdout)
print(f"Return code: {result.returncode}")

export_path = pathlib.Path("book_clean.txt")
print(f"\nExported: {export_path} ({export_path.stat().st_size / 1024:.0f} KB)")
print("\nFirst 500 characters:")
print(export_path.read_text(encoding="utf-8")[:500])

## Expressive mode (requires Claude subscription or API key)

`--expressive` enables a tone-tagging LLM pass that labels each paragraph with
a tone: `neutral`, `somber`, `tense`, `warm`, `wry`, `urgent`, or `reflective`.
These feed into subtle pitch and rate adjustments in Kokoro.

The default backend (`--tone-backend cli`) uses the Claude Code subscription token — no API spend.
The `api` backend uses `VORPAL_ANTHROPIC_KEY` (costs money via the Batches API).

In [ ]:
# Tone tagging only — stop before synthesis so we can inspect the tags
result = subprocess.run(
    ["vorpal", "build", "book.epub", "--expressive", "--tone-backend", "cli",
     "--stop-after", "segment", "--output", "book_expressive"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(f"Return code: {result.returncode}")

# Review the tone labels
result2 = subprocess.run(
    ["vorpal", "review", "book.epub", "--output", "book_expressive", "--tones"],
    capture_output=True, text=True
)
print(result2.stdout[:2000])